<a href="https://colab.research.google.com/github/brajesh995/231FA04C59_MLOPS/blob/main/MLLab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## a. Load a dataset and perform preprocessing

In [19]:
# Load the dataset
df = pd.read_csv('/content/skill_gap_dataset.csv')
display(df.head())

,Skill_Name,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status
0,Java,5,5,10,5,Aligned
1,Python,4,5,10,5,Aligned
2,C++,5,4,8,4,Aligned
3,C,5,4,7,4,Aligned
4,Data Structures,5,5,10,5,Aligned


### 1. Handle missing values

First, let's check for any missing values in the dataset.

In [11]:
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("No missing values found. Proceeding without imputation.")
else:
    print("Missing values detected. Further handling might be required if any columns have missing values.")

Missing values in each column:
Series([], dtype: int64)
No missing values found. Proceeding without imputation.


### 2. Encode categorical variables

In [20]:
from sklearn.preprocessing import OneHotEncoder

# For this dataset, 'Skill_Name' is treated as an identifier and 'Gap_Status' is the target.
# There are no other categorical features that need one-hot encoding for the input X.
# We will create df_encoded by dropping 'Skill_Name' as it's an identifier.
df_encoded = df.drop(columns=['Skill_Name'])

display(df_encoded.head())

,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status
0,5,5,10,5,Aligned
1,4,5,10,5,Aligned
2,5,4,8,4,Aligned
3,5,4,7,4,Aligned
4,5,5,10,5,Aligned


### 3. Normalize/scale data

In [24]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns for scaling
# Exclude 'Gap_Status' as it's the target
numerical_features_to_scale = [col for col in df_encoded.columns if col not in ['Gap_Status']]

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical features
df_scaled = df_encoded.copy()
df_scaled[numerical_features_to_scale] = scaler.fit_transform(df_scaled[numerical_features_to_scale])

display(df_scaled.head())

,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status
0,1.461733,0.916135,2.174936,0.924445,Aligned
1,0.862662,0.916135,2.174936,0.924445,Aligned
2,1.461733,-0.880208,0.188693,-1.001483,Aligned
3,1.461733,-0.880208,-0.804429,-1.001483,Aligned
4,1.461733,0.916135,2.174936,0.924445,Aligned


### 4. Split dataset into training and testing sets

In [25]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
# Drop 'Gap_Status' from X as it's the target. 'Skill_ID' and 'Skill_Name' are no longer in df_scaled.
X = df_scaled.drop(['Gap_Status'], axis=1)
y = df_scaled['Gap_Status']

# Convert target to numerical if it's not already (e.g., 'Aligned' to 0, 'Gap' to 1)
if y.dtype == 'object' or y.dtype == 'bool':
    y = y.astype('category').cat.codes


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (80, 4)
X_test shape: (20, 4)
y_train shape: (80,)
y_test shape: (20,)


### 5. Automate preprocessing steps using a pipeline

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Redefine numerical and categorical features based on the current df structure
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove 'Gap_Status' from numerical_features if it was somehow included (it shouldn't be)
numerical_features = [col for col in numerical_features if col not in ['Gap_Status']]

# For categorical features, remove 'Skill_Name' (identifier) and 'Gap_Status' (target)
categorical_features = [col for col in categorical_features if col not in ['Skill_Name', 'Gap_Status']]

# Create preprocessing pipelines for numerical features
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# If there are no categorical features to encode, the categorical_transformer can be simple or excluded.
# Since we identified no categorical features for X, the list will be empty.
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Create a column transformer to apply different transformations to different columns
# If categorical_features is empty, the 'cat' transformer will effectively do nothing.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features) # This will be applied to an empty list
    ],
    remainder='drop' # Drop any remaining columns not explicitly handled, like 'Skill_Name'
)

# Create a dummy pipeline to demonstrate the preprocessor
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Fit and transform the original data (excluding identifier and target for X_pipeline)
X_pipeline = df.drop(['Skill_Name', 'Gap_Status'], axis=1)
y_pipeline = df['Gap_Status'].astype('category').cat.codes

X_processed_pipeline = pipeline.fit_transform(X_pipeline)

print("Shape of processed data using pipeline:", X_processed_pipeline.shape)
print("First 5 rows of processed data (array format):")
print(X_processed_pipeline[:5])

Shape of processed data using pipeline: (100, 4)
First 5 rows of processed data (array format):
[[ 1.46173315  0.91613482  2.17493636  0.92444546]
 [ 0.86266219  0.91613482  2.17493636  0.92444546]
 [ 1.46173315 -0.88020796  0.18869311 -1.00148258]
 [ 1.46173315 -0.88020796 -0.80442852 -1.00148258]
 [ 1.46173315  0.91613482  2.17493636  0.92444546]]


### 6. Save processed dataset for reuse

In [27]:
import numpy as np

# Get feature names after one-hot encoding for the pipeline output
# This part can be tricky as ColumnTransformer doesn't directly return DataFrame with names
# For simplicity, we'll recreate feature names manually for demonstration

# Fit the preprocessor first if not already done. This is important for get_feature_names_out
pipeline.fit(X_pipeline)

# Get numerical features used in the pipeline
final_numerical_features = pipeline.named_steps['preprocessor'].named_transformers_['num'].get_feature_names_out(numerical_features) if 'num' in pipeline.named_steps['preprocessor'].named_transformers_ else []

# Get one-hot encoded feature names (if any)
onehot_feature_names = []
if 'cat' in pipeline.named_steps['preprocessor'].named_transformers_ and len(categorical_features) > 0: # Check if there were actual categorical features to encode
    onehot_feature_names = pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)

# Combine numerical feature names and one-hot encoded feature names
all_processed_features = list(final_numerical_features)
all_processed_features.extend(onehot_feature_names)

# Convert the processed sparse matrix (if any) or array to a DataFrame
# Handle potential sparse output from OneHotEncoder
if isinstance(X_processed_pipeline, np.ndarray):
    processed_df = pd.DataFrame(X_processed_pipeline, columns=all_processed_features)
else:
    processed_df = pd.DataFrame(X_processed_pipeline.toarray(), columns=all_processed_features)

# Add the target variable back (assuming it was encoded similarly as y_pipeline)
processed_df['Gap_Status'] = y_pipeline.values

# Save to a new CSV file
output_filename = 'skill_gap_dataset_processed.csv'
processed_df.to_csv(output_filename, index=False)
print(f"Processed dataset saved to '{output_filename}'")
display(processed_df.head())

Processed dataset saved to 'skill_gap_dataset_processed.csv'


,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status
0,1.461733,0.916135,2.174936,0.924445,0
1,0.862662,0.916135,2.174936,0.924445,0
2,1.461733,-0.880208,0.188693,-1.001483,0
3,1.461733,-0.880208,-0.804429,-1.001483,0
4,1.461733,0.916135,2.174936,0.924445,0


### 7. Compare results before and after preprocessing

In [28]:
print("\n--- Original DataFrame Info ---")
df.info()
print("\nOriginal DataFrame Head:")
display(df.head())

print("\n--- Processed DataFrame Info (from pipeline) ---")
processed_df.info()
print("\nProcessed DataFrame Head:")
display(processed_df.head())

print("\nComparison of Shapes:")
print(f"Original df shape: {df.shape}")
print(f"Processed df shape: {processed_df.shape}")

print("\nComparison of Data Types (Original vs. Processed):")
print("Original dtypes:\n", df.dtypes.value_counts())
print("Processed dtypes:\n", processed_df.dtypes.value_counts())


--- Original DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Skill_Name           100 non-null    object
 1   Curriculum_Coverage  100 non-null    int64 
 2   Industry_Demand      100 non-null    int64 
 3   Job_Frequency        100 non-null    int64 
 4   Expert_Importance    100 non-null    int64 
 5   Gap_Status           100 non-null    object
dtypes: int64(4), object(2)
memory usage: 4.8+ KB

Original DataFrame Head:


,Skill_Name,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status
0,Java,5,5,10,5,Aligned
1,Python,4,5,10,5,Aligned
2,C++,5,4,8,4,Aligned
3,C,5,4,7,4,Aligned
4,Data Structures,5,5,10,5,Aligned



--- Processed DataFrame Info (from pipeline) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Curriculum_Coverage  100 non-null    float64
 1   Industry_Demand      100 non-null    float64
 2   Job_Frequency        100 non-null    float64
 3   Expert_Importance    100 non-null    float64
 4   Gap_Status           100 non-null    int8   
dtypes: float64(4), int8(1)
memory usage: 3.4 KB

Processed DataFrame Head:


,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status
0,1.461733,0.916135,2.174936,0.924445,0
1,0.862662,0.916135,2.174936,0.924445,0
2,1.461733,-0.880208,0.188693,-1.001483,0
3,1.461733,-0.880208,-0.804429,-1.001483,0
4,1.461733,0.916135,2.174936,0.924445,0



Comparison of Shapes:
Original df shape: (100, 6)
Processed df shape: (100, 5)

Comparison of Data Types (Original vs. Processed):
Original dtypes:
 int64     4
object    2
Name: count, dtype: int64
Processed dtypes:
 float64    4
int8       1
Name: count, dtype: int64
